# 00 — Batch correction with ComBat

| | |
|---|---|
| **runs at** | nobody — this is a lesson |
| **reads** | the three SLE files |
| **writes** | nothing; it changes no file the cycle uses |

**This notebook is not part of the federation cycle.** It exists because batch
correction is the step people get wrong, and it is easier to see on its own than
buried inside a pipeline.

The SLE data has two batches, A with 262 samples and B with 94. They are not just
technically different: batch B holds 44% healthy volunteers against batch A's 17%.
So batch is **entangled with biology**, and that is exactly when correction is
dangerous — remove the batch effect carelessly and you remove the disease effect
with it.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
# The SLE data is not in the repository. Download it from Zenodo
# (doi:10.5281/zenodo.20342569), unpack it, and point SLE at the folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 80        # 80 objects keeps p < n in BOTH cohorts
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
else:
    # A stand-in of the same shape, so every cell runs without the download:
    # two batches of unequal size, a case/control split, and a few proteins
    # measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    FEATURE_MAP = 'feature_metadata.txt'
print(f"{'SLE data' if REAL else 'stand-in'}: n_boot={NBOOT}, top_variable={TOPVAR}")

## Is batch confounded with anything you care about?

Run this **before** correcting, never after. It reports, per variable, a
chi-square *p*-value and a bias-corrected Cramér's V, with a verdict:

- **balanced** — adjust freely
- **partial** — adjustable, but name the variable in `--protect`
- **complete** — every batch holds one level; the design cannot separate them, and
  no method can fix that

The correction is bias-corrected (Bergsma 2013) because the uncorrected form
inflates badly when a variable has many levels: a 96-level plate position once
scored 0.56, "strong", on data whose chi-square *p*-value was 0.46.

In [ ]:
if REAL:
    meta = pd.read_csv(SLE / 'sample-metadata.csv').set_index('SampleId')
    meta = meta[meta['Included_in_study'] == 'Included']
    ab = pd.read_csv(SLE / 'abundance.csv').set_index('SampleId').loc[meta.index]
else:
    ab, meta = build_stand_in()

from pvclust_py.adjust import confounding
print(confounding(meta['Batch'],
                  meta[['Group', 'Sex', 'Disease_activity']]).to_string(index=False))

## What protecting a variable actually does

ComBat estimates, per feature, a location and scale shift for each batch, then
pools those estimates toward a common prior across features (empirical Bayes).
`--protect` puts a variable into the model as a covariate, so its effect is
estimated and **kept** rather than absorbed into the batch term.

Protect the variable that is confounded with batch. Do not protect your outcome
casually in an unsupervised analysis: conditioning on it and then clustering will
find that structure back, which is circular.

In [ ]:
import numpy as np
from pvclust_py.adjust import combat

X = np.log2(ab.astype(float))
keep = X.var(axis=0).sort_values(ascending=False).index[:TOPVAR]
X = X[keep]

corrected = combat(X, meta['Batch'], protect=meta[['Group']].fillna('unknown'))

def batch_gap(M):
    """Mean difference between the two batches, averaged over features."""
    a = M[meta['Batch'] == 'A'].mean(); b = M[meta['Batch'] == 'B'].mean()
    return float((a - b).abs().mean())

print(f'mean |batch A - batch B| per feature')
print(f'  before ComBat: {batch_gap(X):.4f}')
print(f'  after  ComBat: {batch_gap(corrected):.4f}')

## Did it keep the biology?

The number that matters is not how much the batch gap shrank — that is easy to
drive to zero by destroying the data. It is whether the **group** difference
survived. If correction shrinks both, it removed signal along with noise.

In [ ]:
def group_gap(M):
    hv = M[meta['Group'] == 'HV'].mean(); sle = M[meta['Group'] == 'SLE'].mean()
    return float((hv - sle).abs().mean())

print(f'mean |HV - SLE| per feature')
print(f'  before ComBat: {group_gap(X):.4f}')
print(f'  after  ComBat: {group_gap(corrected):.4f}')
print()
print('batch gap should fall a lot; group gap should not.')

---
**Why this is a no-op in the federation cycle.** Cohort A *is* batch A and cohort B
*is* batch B, so neither cohort holds more than one batch and neither has anything
to correct against locally. The code path stays in the cohort notebooks because a
real cohort usually spans several plates.

Correcting across both batches before splitting would use pooled data, which a real
federation cannot do. Federating the correction itself — each cohort shipping
per-feature means, variances and counts, the aggregator returning each cohort its
own shift and scale — is the right answer and is **not built**.